# 🌿 The Green-Truth Auditor: Combating Greenwashing
### Theme: Responsible Consumption & Climate Action

**Problem:** Brands use vague marketing buzzwords to hide high-carbon supply chains.

**Solution:** An Intent-Aware Sustainability Auditor that:
- Accepts product descriptions (text box) **or** a URL (web scraper)
- Flags buzzwords with a rule-based engine
- Classifies each sentence as `Marketing Fluff` vs `Evidence-Based` using a trained ML model
- Checks brand against a RAG database of certified B-Corp / GOTS brands
- Generates an AI reasoning summary explaining exactly why a product failed

---
**Covers all Must-Haves + all Good-to-Haves from the problem statement.**

##  Step 0 — Install Dependencies

In [27]:
# Install all required libraries
!pip install -q datasets sentence-transformers faiss-cpu scikit-learn \
              pandas numpy requests beautifulsoup4 transformers torch \
              ipywidgets

In [28]:
import re
import json
import csv
import io
import warnings
import textwrap
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import requests
import faiss

from bs4 import BeautifulSoup
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

print(' All imports successful')

 All imports successful


##  Step 1 — Load & Train on the Greenwashing Dataset

In [29]:
# ── Load HuggingFace dataset ───────────────────────────────────────────────────
raw = load_dataset('Emanuse/greenwashing')
print('Splits available:', list(raw.keys()))

split_name = 'train' if 'train' in raw else list(raw.keys())[0]
df = raw[split_name].to_pandas()

print(f'Shape: {df.shape}')
print('Columns:', df.columns.tolist())
print('\nSample rows:')
df.head(4)

Splits available: ['train']
Shape: (2163, 1)
Columns: ['text']

Sample rows:


,text
0,ACCELERATING
1,MEANINGFUL
2,CHANGE
3,2018 Sustainability Report


In [40]:
# ── Auto-detect text and label columns ────────────────────────────────────────
text_col  = [c for c in df.columns if df[c].dtype == object][0]
label_col = [c for c in df.columns if c != text_col][0]
df = df.rename(columns={text_col: 'text', label_col: 'label'})

print(f'Text column: {text_col!r}   Label column: {label_col!r}')
print('Label distribution:')
print(df['label'].value_counts())

Text column: 'text'   Label column: 'label'
Label distribution:
label
Marketing Fluff    1698
Evidence-Based      412
Name: count, dtype: int64


In [36]:
# ── Text preprocessing ─────────────────────────────────────────────────────────
def preprocess_text(text: str) -> str:
    """Cleans text while preserving % and numbers — key signals for greenwashing detection."""
    if not isinstance(text, str):
        return ''
    text = text.lower().strip()
    text = re.sub(r'http\S+|www\.\S+', '', text)        # strip URLs
    text = re.sub(r'[^a-z0-9\s%\-]', ' ', text)         # keep alphanum + % + hyphen
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(preprocess_text)
df = df[df['clean_text'].str.len() > 5].reset_index(drop=True)
print(f'Dataset after cleaning: {len(df)} rows')

Dataset after cleaning: 2110 rows


In [42]:
# Encode labels
le = LabelEncoder()
y = le.fit_transform(df['label'])
X = df['text'].values

# Train/test split (no stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# TF-IDF + Logistic Regression pipeline
ml_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1,2), max_features=10000, sublinear_tf=True, min_df=2, strip_accents='unicode'
    )),
    ('clf', LogisticRegression(
        max_iter=500, C=1.0, class_weight='balanced', random_state=42
    ))
])

ml_pipeline.fit(X_train, y_train)
y_pred = ml_pipeline.predict(X_test)

print(f" Model trained on {len(X_train)} samples")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

 Model trained on 1688 samples
Test Accuracy: 0.924

Classification Report:
                 precision    recall  f1-score   support

 Evidence-Based       0.82      0.73      0.77        73
Marketing Fluff       0.94      0.97      0.95       349

       accuracy                           0.92       422
      macro avg       0.88      0.85      0.86       422
   weighted avg       0.92      0.92      0.92       422



##  Step 2 — Buzzword Detection (Rule-Based Flag System)

> Add blockquote



In [43]:
# ── Greenwashing buzzword list (expandable) ────────────────────────────────────
#
# These terms are flagged per the problem statement:
#   "natural", "green", "eco-conscious", "eco-friendly", etc.
# ─────────────────────────────────────────────────────────────────────────────

BUZZWORDS = [
    # Vague eco claims
    r'\beco[- ]?friendly\b', r'\beco[- ]?conscious\b', r'\beco[- ]?smart\b',
    r'\bgreen\b', r'\bgo(?:ing)?\s+green\b', r'\bsustainable\b',
    r'\bnatural\b', r'\ball[- ]?natural\b', r'\b100\s*%\s*natural\b',
    r'\bclean\b', r'\bpure\b', r'\bnon[- ]?toxic\b',
    r'\bplanet[- ]?friendly\b', r'\bearth[- ]?friendly\b',
    r'\bresponsible\b', r'\bethical\b', r'\bconscious\b',
    r'\bsave the planet\b', r'\bcare(?:s)? about the planet\b',
    r'\bkinder to the earth\b', r'\bbetter for the environment\b',
    r'\bwe love the earth\b', r'\bthink(?:ing)? green\b',
    r'\bclimate[- ]?friendly\b', r'\bzero[- ]?waste\b(?!\s+certified)',
    r'\bcarbon[- ]?friendly\b', r'\bgreen(?:er)?\s+future\b',
    r'\bmade with love for the planet\b', r'\bplanet first\b',
]

# Positive evidence patterns (NOT greenwashing)
EVIDENCE_PATTERNS = {
    'percentage'  : re.compile(r'\b(\d+(\.\d+)?\s*%)'),
    'quantity'    : re.compile(r'\b\d{2,}\s*(tons?|kg|kwh|mwh|tonnes?|lbs?)\b'),
    'year_target' : re.compile(r'\bby\s+20[2-5]\d\b'),
    'since_year'  : re.compile(r'\bsince\s+(?:19|20)\d{2}\b'),
    'third_party' : re.compile(
        r'\b(certified|accredited|verified|audited)\s+by\b', re.I
    ),
}

def detect_buzzwords(text: str) -> dict:
    """
    Returns:
      - flagged_buzzwords : list of vague terms found
      - evidence_signals  : dict of positive evidence found
      - buzzword_count    : int
    """
    text_lower = text.lower()
    flagged = []
    for pattern in BUZZWORDS:
        hits = re.findall(pattern, text_lower)
        if hits:
            # Extract the matched string cleanly
            flagged.extend([h if isinstance(h, str) else h[0] for h in hits])

    evidence = {}
    for sig_name, sig_re in EVIDENCE_PATTERNS.items():
        found = sig_re.findall(text_lower)
        if found:
            evidence[sig_name] = [f[0] if isinstance(f, tuple) else f for f in found]

    return {
        'flagged_buzzwords' : list(set(flagged)),
        'buzzword_count'    : len(set(flagged)),
        'evidence_signals'  : evidence,
        'has_evidence'      : bool(evidence),
    }


# ── Quick test ────────────────────────────────────────────────────────────────
sample = 'Our eco-friendly and natural product is 100% sustainable. Certified by USDA Organic. Reduced CO2 by 40% since 2019.'
result = detect_buzzwords(sample)
print('Buzzwords flagged :', result['flagged_buzzwords'])
print('Evidence found    :', result['evidence_signals'])

Buzzwords flagged : ['natural', 'sustainable', 'eco-friendly']
Evidence found    : {'percentage': ['100%', '40%'], 'since_year': ['since 2019'], 'third_party': ['certified']}


##  Step 3 — RAG: Certified Brand Database (B-Corp / GOTS)

In [44]:
# ── Certified Brand Database (CSV-style, easily replaceable with a real CSV) ──

CERTIFIED_BRANDS_CSV = """
brand,certification,category,description
Patagonia,B-Corp,Outdoor Apparel,Outdoor clothing brand committed to environmental activism and fair trade
Eileen Fisher,B-Corp,Fashion,Sustainable fashion brand with take-back recycling program
Allbirds,B-Corp,Footwear,Carbon-neutral footwear made from natural materials like merino wool
Seventh Generation,B-Corp,Household Products,Plant-based cleaning and personal care products
Ben & Jerry's,B-Corp,Food,Ice cream company with strong social and environmental commitments
Danone,B-Corp,Food & Beverage,Global food company with certified sustainable operations
Natura,B-Corp,Cosmetics,Brazilian cosmetics company committed to Amazon biodiversity
Interface,B-Corp,Manufacturing,Carpet tile manufacturer with Mission Zero carbon commitment
Cotopaxi,B-Corp,Outdoor Gear,Gear brand donating 1% revenue to poverty alleviation
Bombas,B-Corp,Apparel,Sock company donating one item for every item purchased
Tentree,GOTS,Fashion,Plants ten trees for every item purchased; organic cotton
Pact,GOTS,Fashion,Organic cotton basics certified GOTS and Fair Trade
Thought Clothing,GOTS,Fashion,UK sustainable clothing from organic bamboo and cotton
People Tree,GOTS,Fashion,Pioneer of fair trade and organic fashion
Coyuchi,GOTS,Home Textiles,Organic cotton bedding and bath products
Trusted Clothes,GOTS,Fashion,Platform connecting consumers to certified sustainable brands
Avocado,GOTS,Mattresses,Organic mattresses certified GOTS and GREENGUARD Gold
Boll & Branch,GOTS,Bedding,Fair trade and GOTS certified organic cotton bedding
Ecocert,EcoCert,Cosmetics,Certifying body for organic and natural cosmetics
Dr. Bronner's,Fair Trade + B-Corp,Personal Care,Fair trade and organic certified personal care products
Veja,Fair Trade,Footwear,Transparent fair trade sneakers made with Amazonian rubber
Tony's Chocolonely,Fair Trade,Food,Bean-to-bar chocolate committed to 100% slave-free cocoa
Fjallraven,Bluesign,Outdoor Apparel,Swedish outdoor brand using bluesign certified materials
Nudie Jeans,GOTS,Denim,Swedish denim brand with 100% organic cotton and repair culture
Weleda,Natrue,Cosmetics,Natural and organic certified cosmetics and personal care
"""

brand_df = pd.read_csv(io.StringIO(CERTIFIED_BRANDS_CSV.strip()))
print(f'Loaded {len(brand_df)} certified brands into database')
brand_df.head(6)

Loaded 25 certified brands into database


,brand,certification,category,description
0,Patagonia,B-Corp,Outdoor Apparel,Outdoor clothing brand committed to environmen...
1,Eileen Fisher,B-Corp,Fashion,Sustainable fashion brand with take-back recyc...
2,Allbirds,B-Corp,Footwear,Carbon-neutral footwear made from natural mate...
3,Seventh Generation,B-Corp,Household Products,Plant-based cleaning and personal care products
4,Ben & Jerry's,B-Corp,Food,Ice cream company with strong social and envir...
5,Danone,B-Corp,Food & Beverage,Global food company with certified sustainable...


In [45]:
# ── Build FAISS semantic index over brand descriptions ─────────────────────────
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# Embed: brand name + certification + description
brand_texts = (
    brand_df['brand'] + ' ' +
    brand_df['certification'] + ' ' +
    brand_df['description']
).tolist()

brand_embeddings = embed_model.encode(brand_texts, show_progress_bar=False)
brand_embeddings /= np.linalg.norm(brand_embeddings, axis=1, keepdims=True)  # cosine

dim = brand_embeddings.shape[1]
brand_index = faiss.IndexFlatIP(dim)  # Inner Product = cosine on normalized vecs
brand_index.add(brand_embeddings.astype('float32'))

print(f' FAISS index ready: {brand_index.ntotal} certified brands indexed (dim={dim})')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 FAISS index ready: 25 certified brands indexed (dim=384)


In [46]:
# ── RAG Query Function ─────────────────────────────────────────────────────────
def query_certified_brands(text: str, top_k: int = 3, threshold: float = 0.30) -> list[dict]:
    """
    Checks if the input text mentions or semantically matches any certified brand.
    Uses two-pass: (1) exact brand name match, (2) FAISS semantic similarity.
    Returns list of matching certified brands with their details.
    """
    text_lower = text.lower()
    matches = []

    # Pass 1: Exact brand name lookup (highest precision)
    for _, row in brand_df.iterrows():
        if row['brand'].lower() in text_lower:
            matches.append({
                'brand'        : row['brand'],
                'certification': row['certification'],
                'category'     : row['category'],
                'match_type'   : 'exact_name',
                'similarity'   : 1.0
            })

    exact_names = {m['brand'] for m in matches}

    # Pass 2: Semantic similarity (catches "the outdoor gear company from California" etc.)
    vec = embed_model.encode([text], show_progress_bar=False)
    vec /= np.linalg.norm(vec, axis=1, keepdims=True)
    scores, indices = brand_index.search(vec.astype('float32'), top_k)

    for sim, idx in zip(scores[0], indices[0]):
        row = brand_df.iloc[idx]
        if float(sim) >= threshold and row['brand'] not in exact_names:
            matches.append({
                'brand'        : row['brand'],
                'certification': row['certification'],
                'category'     : row['category'],
                'match_type'   : 'semantic',
                'similarity'   : round(float(sim), 3)
            })

    return matches


# Smoke test
print(query_certified_brands('This Patagonia jacket is great for hiking'))
print(query_certified_brands('sustainable outdoor gear for adventurers'))

[{'brand': 'Patagonia', 'certification': 'B-Corp', 'category': 'Outdoor Apparel', 'match_type': 'exact_name', 'similarity': 1.0}]
[{'brand': 'Thought Clothing', 'certification': 'GOTS', 'category': 'Fashion', 'match_type': 'semantic', 'similarity': 0.463}, {'brand': 'Patagonia', 'certification': 'B-Corp', 'category': 'Outdoor Apparel', 'match_type': 'semantic', 'similarity': 0.445}, {'brand': 'Trusted Clothes', 'certification': 'GOTS', 'category': 'Fashion', 'match_type': 'semantic', 'similarity': 0.395}]


##  Step 4 — Input Interface: Text Box + URL Scraper

In [47]:
# ── URL Scraper ────────────────────────────────────────────────────────────────
def scrape_url(url: str, timeout: int = 10) -> str:
    """
    Scrapes visible text from a product page URL.
    Extracts: title, meta description, paragraphs, list items.
    Returns cleaned plain text (or raises an informative error).
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (compatible; GreenTruthAuditor/1.0)'
    }

    try:
        response = requests.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f'Failed to fetch URL: {e}')

    soup = BeautifulSoup(response.text, 'html.parser')

    # Remove boilerplate tags
    for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside', 'noscript']):
        tag.decompose()

    # Collect text
    parts = []

    title = soup.find('title')
    if title:
        parts.append(title.get_text(strip=True))

    meta = soup.find('meta', attrs={'name': 'description'})
    if meta and meta.get('content'):
        parts.append(meta['content'])

    for tag in soup.find_all(['p', 'li', 'h1', 'h2', 'h3', 'span']):
        text = tag.get_text(separator=' ', strip=True)
        if len(text) > 20:           # skip trivially short fragments
            parts.append(text)

    combined = ' '.join(parts)
    # Clean up excessive whitespace
    combined = re.sub(r'\s+', ' ', combined).strip()

    if not combined:
        raise RuntimeError('Could not extract text from the page. It may be JavaScript-rendered.')

    return combined[:5000]  # cap at 5000 chars to stay within context


def get_input_text(text_input: str = '', url_input: str = '') -> str:
    """
    Unified input handler.
    Priority: URL (scrapes page) > direct text input.
    Returns the raw text to be audited.
    """
    if url_input.strip():
        print(f'🌐 Scraping URL: {url_input}')
        text = scrape_url(url_input)
        print(f'   Extracted {len(text)} characters from page')
        return text
    elif text_input.strip():
        return text_input.strip()
    else:
        raise ValueError('Provide either text_input or url_input.')


print(' URL scraper and input handler ready')

 URL scraper and input handler ready


##  Step 5 — AI Reasoning Summary Generator (Good-to-Have)

> Add blockquote



In [48]:
# ── AI Reasoning Summary ───────────────────────────────────────────────────────
#
# Generates a human-readable explanation of WHY a product failed (or passed)
# the audit. Uses rule-based template reasoning — no API key needed.
# In production: swap generate_reasoning() body with an LLM API call.
# ─────────────────────────────────────────────────────────────────────────────

def generate_reasoning(
    score: int,
    verdict: str,
    buzzwords: list,
    evidence: dict,
    certified_brands: list,
    fluff_sentences: list,
    evidence_sentences: list,
) -> str:
    """
    Returns a structured, plain-English audit explanation.
    """
    lines = []
    lines.append(f'AUDIT VERDICT: {verdict}  (Trust Score: {score}/100)')
    lines.append('=' * 60)

    # ── Why it failed / succeeded ──────────────────────────────────────────
    if score < 45:
        lines.append(
            '\n  FAIL REASON: This product description relies heavily on '
            'vague, unverifiable language that is a hallmark of greenwashing. '
            'No concrete data, third-party certifications, or measurable claims '
            'were found to substantiate the environmental assertions made.'
        )
    elif score < 70:
        lines.append(
            '\n MIXED RESULT: The description contains some evidence-based '
            'claims but also relies on unverifiable marketing language. '
            'Independent verification is recommended before trusting these claims.'
        )
    else:
        lines.append(
            '\n PASS REASON: The description contains verifiable claims, '
            'measurable data, or recognized certifications that support its '
            'sustainability assertions.'
        )

    # ── Buzzword analysis ──────────────────────────────────────────────────
    if buzzwords:
        lines.append(f'\n VAGUE BUZZWORDS DETECTED ({len(buzzwords)}):')
        for bw in buzzwords[:8]:   # cap display at 8
            lines.append(f'   • "{bw}" — unverifiable without third-party proof')
        if len(buzzwords) > 8:
            lines.append(f'   ... and {len(buzzwords)-8} more')
    else:
        lines.append('\n No greenwashing buzzwords detected.')

    # ── Evidence analysis ──────────────────────────────────────────────────
    if evidence:
        lines.append('\n POSITIVE EVIDENCE FOUND:')
        if 'percentage' in evidence:
            lines.append(f'   • Percentage claims: {evidence["percentage"]} — quantified, verifiable')
        if 'quantity' in evidence:
            lines.append(f'   • Quantity claims: {evidence["quantity"]} — specific measurable impact')
        if 'year_target' in evidence:
            lines.append(f'   • Time-bound commitment: {evidence["year_target"]} — holds brand accountable')
        if 'third_party' in evidence:
            lines.append('   • Third-party verification language detected')
    else:
        lines.append('\n No quantified or verifiable evidence found in text.')

    # ── Certified brand match ──────────────────────────────────────────────
    if certified_brands:
        lines.append('\n CERTIFIED BRAND MATCH:')
        for b in certified_brands[:3]:
            lines.append(
                f'   • {b["brand"]} ({b["certification"]}) — '
                f'recognized certified brand in our database [{b["match_type"]}]'
            )
    else:
        lines.append(
            '\n BRAND CHECK: No recognized certified brand (B-Corp/GOTS/Fair Trade) '
            'was identified in this description. Lack of brand transparency is itself a red flag.'
        )

    # ── Sentence-level breakdown ───────────────────────────────────────────
    if fluff_sentences:
        lines.append('\n FLAGGED SENTENCES (Marketing Fluff):')
        for s in fluff_sentences[:3]:
            lines.append(f'   → "{s[:100]}..."' if len(s) > 100 else f'   → "{s}"')

    if evidence_sentences:
        lines.append('\n CREDIBLE SENTENCES (Evidence-Based):')
        for s in evidence_sentences[:3]:
            lines.append(f'   → "{s[:100]}..."' if len(s) > 100 else f'   → "{s}"')

    # ── Recommendation ─────────────────────────────────────────────────────
    lines.append('\n💡 RECOMMENDATION:')
    if score < 45:
        lines.append(
            '   Look for products certified by recognized bodies (B-Corp, GOTS, '
            'Fair Trade, FSC, USDA Organic). Ask brands for third-party audit reports.'
        )
    elif score < 70:
        lines.append(
            '   Request third-party certification documentation. The brand shows '
            'some transparency but has not fully substantiated all claims.'
        )
    else:
        lines.append(
            '   This product appears to have legitimate sustainability credentials. '
            'Still verify certifications directly on the certifying body website.'
        )

    return '\n'.join(lines)


print(' Reasoning generator ready')

 Reasoning generator ready


##  Step 6 — Full Scoring Pipeline

In [49]:
# ── Scoring weights (calibrated) ───────────────────────────────────────────────
WEIGHTS = {
    'base'              :  50,
    'ml_evidence'       :  +8,   # model says Evidence-Based
    'ml_fluff'          : -10,   # model says Marketing Fluff
    'high_confidence'   :  +4,   # model confidence ≥ 0.80
    'cert_brand_exact'  : +15,   # exact certified brand name in text
    'cert_brand_sem'    :  +7,   # semantically matched certified brand
    'percentage'        :  +6,   # quantified % claim
    'quantity'          :  +5,   # quantity with units
    'year_target'       :  +4,   # time-bound commitment
    'third_party'       :  +6,   # "certified by" language
    'buzzword_penalty'  :  -4,   # per unique buzzword (capped)
    'max_buzz_penalty'  : -20,   # max total buzzword penalty
}


def sentence_level_classify(text: str) -> tuple[list, list]:
    """
    Splits text into sentences, classifies each.
    Returns (fluff_sentences, evidence_sentences).
    """
    # Simple sentence splitter
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if len(s.strip()) > 15]

    fluff_sents    = []
    evidence_sents = []

    for sent in sentences:
        clean = preprocess_text(sent)
        if not clean:
            continue
        proba   = ml_pipeline.predict_proba([clean])[0]
        pred_i  = int(np.argmax(proba))
        label   = le.inverse_transform([pred_i])[0]

        label_lower = label.lower()
        if any(k in label_lower for k in ['fluff', 'marketing', 'vague', '0']):
            fluff_sents.append(sent)
        else:
            evidence_sents.append(sent)

    return fluff_sents, evidence_sents


def compute_trust_score(ml_label, confidence, buzz, rag_brands, evidence) -> tuple[int, str]:
    """Computes 0-100 trust score and returns (score, brief_breakdown)."""
    score = WEIGHTS['base']
    log   = [f'Base: {score}']

    label_lower = ml_label.lower()
    if any(k in label_lower for k in ['evidence', 'fact', 'verified', '1']):
        score += WEIGHTS['ml_evidence']
        log.append(f"ML→Evidence-Based (+{WEIGHTS['ml_evidence']})")
    else:
        score += WEIGHTS['ml_fluff']
        log.append(f"ML→Marketing Fluff ({WEIGHTS['ml_fluff']})")

    if confidence >= 0.80:
        score += WEIGHTS['high_confidence']
        log.append(f"High confidence (+{WEIGHTS['high_confidence']})")

    for brand in rag_brands:
        if brand['match_type'] == 'exact_name':
            score += WEIGHTS['cert_brand_exact']
            log.append(f"{brand['brand']} certified brand match (+{WEIGHTS['cert_brand_exact']})")
        else:
            score += WEIGHTS['cert_brand_sem']
            log.append(f"{brand['brand']} semantic match (+{WEIGHTS['cert_brand_sem']})")

    ev = evidence.get('evidence_signals', {})
    if ev.get('percentage'):
        score += WEIGHTS['percentage']
        log.append(f"Percentage claim (+{WEIGHTS['percentage']})")
    if ev.get('quantity'):
        score += WEIGHTS['quantity']
        log.append(f"Quantity claim (+{WEIGHTS['quantity']})")
    if ev.get('year_target'):
        score += WEIGHTS['year_target']
        log.append(f"Time-bound target (+{WEIGHTS['year_target']})")
    if ev.get('third_party'):
        score += WEIGHTS['third_party']
        log.append(f"Third-party language (+{WEIGHTS['third_party']})")

    n_buzz = buzz['buzzword_count']
    if n_buzz > 0:
        penalty = min(n_buzz * abs(WEIGHTS['buzzword_penalty']), abs(WEIGHTS['max_buzz_penalty']))
        score  -= penalty
        log.append(f"{n_buzz} buzzwords (-{penalty})")

    score = max(0, min(100, score))
    return score, ' | '.join(log)


print(' Scoring pipeline ready')

 Scoring pipeline ready


##  Step 7 — `audit()` — The Complete Green-Truth Auditor

In [50]:
def audit(text_input: str = '', url_input: str = '', verbose: bool = True) -> dict:
    """
    The Green-Truth Auditor.

    Inputs:
      text_input : product description text
      url_input  : product page URL (scrapes automatically)
      verbose    : if True, prints the full audit report

    Returns a dict with:
      - classification       : Marketing Fluff / Evidence-Based
      - confidence           : ML model confidence
      - trust_score          : 0-100
      - verdict              : 🚩 / ⚠️ / ✅
      - flagged_buzzwords    : list of vague terms
      - evidence_signals     : dict of positive evidence
      - certified_brands     : matching brands from RAG database
      - fluff_sentences      : sentences classified as fluff
      - evidence_sentences   : sentences classified as evidence-based
      - score_breakdown      : how the score was computed
      - reasoning_summary    : AI-generated audit explanation
    """
    # ── 1. Input ────────────────────────────────────────────────────────────
    raw_text = get_input_text(text_input, url_input)

    # ── 2. Preprocess ───────────────────────────────────────────────────────
    clean = preprocess_text(raw_text)

    # ── 3. ML Classification (whole-text) ───────────────────────────────────
    proba      = ml_pipeline.predict_proba([clean])[0]
    pred_idx   = int(np.argmax(proba))
    confidence = float(np.max(proba))
    ml_label   = le.inverse_transform([pred_idx])[0]

    # ── 4. Sentence-level classification ───────────────────────────────────
    fluff_sents, evidence_sents = sentence_level_classify(raw_text)

    # ── 5. Buzzword detection ───────────────────────────────────────────────
    buzz = detect_buzzwords(raw_text)

    # ── 6. RAG: Certified brand lookup ─────────────────────────────────────
    cert_brands = query_certified_brands(raw_text)

    # ── 7. Trust score ──────────────────────────────────────────────────────
    score, breakdown = compute_trust_score(ml_label, confidence, buzz, cert_brands, buzz)

    if score >= 70:
        verdict = '✅ LEGITIMATE — Claims appear substantiated'
    elif score >= 45:
        verdict = '⚠️  UNCERTAIN — Partially verifiable, needs scrutiny'
    else:
        verdict = '🚩 GREENWASHING — Vague, unverifiable claims detected'

    # ── 8. AI Reasoning Summary ─────────────────────────────────────────────
    reasoning = generate_reasoning(
        score, verdict, buzz['flagged_buzzwords'],
        buzz['evidence_signals'], cert_brands,
        fluff_sents, evidence_sents
    )

    result = {
        'classification'    : ml_label,
        'confidence'        : round(confidence, 3),
        'trust_score'       : score,
        'verdict'           : verdict,
        'flagged_buzzwords' : buzz['flagged_buzzwords'],
        'evidence_signals'  : buzz['evidence_signals'],
        'certified_brands'  : cert_brands,
        'fluff_sentences'   : fluff_sents,
        'evidence_sentences': evidence_sents,
        'score_breakdown'   : breakdown,
        'reasoning_summary' : reasoning,
    }

    if verbose:
        print(reasoning)
        print(f'\n📐 Score breakdown: {breakdown}')

    return result


print(' audit() function ready — use audit(text_input=...) or audit(url_input=...)')

 audit() function ready — use audit(text_input=...) or audit(url_input=...)


##  Step 8 — Demo: Test All 4 Cases

In [55]:
# ─────────────────────────────────────────────────────────────────────────────
# CASE 1: Pure greenwashing — all buzzwords, zero evidence
# ─────────────────────────────────────────────────────────────────────────────
print('\n' + '='*70)
print('CASE 1: Pure Greenwashing')
print('='*70)

r1 = audit(
    text_input=(
        "Our revolutionary product is built for conscious consumers who care deeply about the planet. "
        "Designed with nature in mind, it supports a healthier lifestyle and promotes environmental harmony. "
        "Crafted using innovative green technology and holistic design principles, it represents the future of sustainable living."
    )
)

print(r1)


CASE 1: Pure Greenwashing
AUDIT VERDICT: ⚠️  UNCERTAIN — Partially verifiable, needs scrutiny  (Trust Score: 53/100)

 MIXED RESULT: The description contains some evidence-based claims but also relies on unverifiable marketing language. Independent verification is recommended before trusting these claims.

 VAGUE BUZZWORDS DETECTED (3):
   • "green" — unverifiable without third-party proof
   • "sustainable" — unverifiable without third-party proof
   • "conscious" — unverifiable without third-party proof

 No quantified or verifiable evidence found in text.

 CERTIFIED BRAND MATCH:
   • Trusted Clothes (GOTS) — recognized certified brand in our database [semantic]
   • Patagonia (B-Corp) — recognized certified brand in our database [semantic]
   • Danone (B-Corp) — recognized certified brand in our database [semantic]

 FLAGGED SENTENCES (Marketing Fluff):
   → "Our revolutionary product is built for conscious consumers who care deeply about the planet."
   → "Designed with nature in

In [56]:
# ─────────────────────────────────────────────────────────────────────────────
# CASE 2: Evidence-based — data, certifications, time targets
# ─────────────────────────────────────────────────────────────────────────────
print('\n' + '='*70)
print('CASE 2: Evidence-Based Claims')
print('='*70)
r2 = audit(
    text_input=(
        'Certified B-Corp since 2014. Our denim is made with 70% recycled '
        'cotton, certified by GOTS. We reduced water usage by 50% per garment '
        'since 2019. Net-zero carbon emissions target by 2030. Audited by '
        'Bureau Veritas annually.'
    )
)


CASE 2: Evidence-Based Claims
AUDIT VERDICT: ✅ LEGITIMATE — Claims appear substantiated  (Trust Score: 99/100)

 PASS REASON: The description contains verifiable claims, measurable data, or recognized certifications that support its sustainability assertions.

 No greenwashing buzzwords detected.

 POSITIVE EVIDENCE FOUND:
   • Percentage claims: ['70%', '50%'] — quantified, verifiable
   • Time-bound commitment: ['by 2030'] — holds brand accountable
   • Third-party verification language detected

 CERTIFIED BRAND MATCH:
   • Nudie Jeans (GOTS) — recognized certified brand in our database [semantic]
   • Thought Clothing (GOTS) — recognized certified brand in our database [semantic]
   • Boll & Branch (GOTS) — recognized certified brand in our database [semantic]

 CREDIBLE SENTENCES (Evidence-Based):
   → "Certified B-Corp since 2014."
   → "Our denim is made with 70% recycled cotton, certified by GOTS."
   → "We reduced water usage by 50% per garment since 2019."

💡 RECOMMENDATION:

In [57]:
# ─────────────────────────────────────────────────────────────────────────────
# CASE 3: Mixed — some claims, some buzzwords
# ─────────────────────────────────────────────────────────────────────────────
print('\n' + '='*70)
print('CASE 3: Mixed (Uncertain)')
print('='*70)
r3 = audit(
    text_input=(
        'Our natural and eco-conscious packaging uses 40% less plastic. '
        'We are committed to a greener future by 2025. No certification '
        'has been obtained yet but we are working toward it.'
    )
)


CASE 3: Mixed (Uncertain)
AUDIT VERDICT: ✅ LEGITIMATE — Claims appear substantiated  (Trust Score: 70/100)

 PASS REASON: The description contains verifiable claims, measurable data, or recognized certifications that support its sustainability assertions.

 VAGUE BUZZWORDS DETECTED (4):
   • "eco-conscious" — unverifiable without third-party proof
   • "natural" — unverifiable without third-party proof
   • "conscious" — unverifiable without third-party proof
   • "greener future" — unverifiable without third-party proof

 POSITIVE EVIDENCE FOUND:
   • Percentage claims: ['40%'] — quantified, verifiable
   • Time-bound commitment: ['by 2025'] — holds brand accountable

 CERTIFIED BRAND MATCH:
   • Natura (B-Corp) — recognized certified brand in our database [exact_name]
   • Ecocert (EcoCert) — recognized certified brand in our database [semantic]
   • Trusted Clothes (GOTS) — recognized certified brand in our database [semantic]

 FLAGGED SENTENCES (Marketing Fluff):
   → "We are com

In [58]:
# ─────────────────────────────────────────────────────────────────────────────
# CASE 4: Known certified brand (Patagonia)
# ─────────────────────────────────────────────────────────────────────────────
print('\n' + '='*70)
print('CASE 4: Known B-Corp Brand (Patagonia)')
print('='*70)
r4 = audit(
    text_input=(
        'Patagonia Nano Puff Jacket. Made with 60g PrimaLoft Gold Insulation '
        'Eco — 55% recycled post-consumer content. Patagonia is a certified '
        'B Corporation. We donate 1% of all sales to environmental groups.'
    )
)


CASE 4: Known B-Corp Brand (Patagonia)
AUDIT VERDICT: ✅ LEGITIMATE — Claims appear substantiated  (Trust Score: 75/100)

 PASS REASON: The description contains verifiable claims, measurable data, or recognized certifications that support its sustainability assertions.

 No greenwashing buzzwords detected.

 POSITIVE EVIDENCE FOUND:
   • Percentage claims: ['55%', '1%'] — quantified, verifiable

 CERTIFIED BRAND MATCH:
   • Patagonia (B-Corp) — recognized certified brand in our database [exact_name]
   • Eileen Fisher (B-Corp) — recognized certified brand in our database [semantic]
   • Allbirds (B-Corp) — recognized certified brand in our database [semantic]

 FLAGGED SENTENCES (Marketing Fluff):
   → "Patagonia Nano Puff Jacket."
   → "Made with 60g PrimaLoft Gold Insulation Eco — 55% recycled post-consumer content."
   → "Patagonia is a certified B Corporation."

💡 RECOMMENDATION:
   This product appears to have legitimate sustainability credentials. Still verify certifications dire

In [59]:
# ─────────────────────────────────────────────────────────────────────────────
# CASE 5: URL input (live scrape)
# Replace with any product URL — this uses a real scraper.
# ─────────────────────────────────────────────────────────────────────────────
url_to_test = 'https://www.allbirds.com/pages/sustainable-practices'  # example

print('\n' + '='*70)
print(f'CASE 5: URL Scrape — {url_to_test}')
print('='*70)

try:
    r5 = audit(url_input=url_to_test)
except Exception as e:
    print(f'⚠️  Could not scrape URL (expected in some Colab envs): {e}')
    print('   → Use audit(text_input="...") for text-based auditing.')


CASE 5: URL Scrape — https://www.allbirds.com/pages/sustainable-practices
🌐 Scraping URL: https://www.allbirds.com/pages/sustainable-practices
   Extracted 4043 characters from page
AUDIT VERDICT: ✅ LEGITIMATE — Claims appear substantiated  (Trust Score: 95/100)

 PASS REASON: The description contains verifiable claims, measurable data, or recognized certifications that support its sustainability assertions.

 VAGUE BUZZWORDS DETECTED (2):
   • "sustainable" — unverifiable without third-party proof
   • "natural" — unverifiable without third-party proof

 POSITIVE EVIDENCE FOUND:

 CERTIFIED BRAND MATCH:
   • Allbirds (B-Corp) — recognized certified brand in our database [exact_name]
   • Natura (B-Corp) — recognized certified brand in our database [exact_name]
   • Pact (GOTS) — recognized certified brand in our database [exact_name]

 FLAGGED SENTENCES (Marketing Fluff):
   → "Sustainability Guide & Practices | Allbirds At Allbirds, sustainability is at the foundation of our ..."
  

In [61]:
# ─────────────────────────────────────────────────────────────────────────────
# CASE 5: URL input (live scrape)
# Replace with any product URL — this uses a real scraper.
# ─────────────────────────────────────────────────────────────────────────────
url_to_test = 'https://www.apple.com/environment/'

print('\n' + '='*70)
print(f'CASE 5: URL Scrape — {url_to_test}')
print('='*70)

try:
    r5 = audit(url_input=url_to_test)
except Exception as e:
    print(f'⚠️  Could not scrape URL (expected in some Colab envs): {e}')
    print('   → Use audit(text_input="...") for text-based auditing.')


CASE 5: URL Scrape — https://www.apple.com/environment/
🌐 Scraping URL: https://www.apple.com/environment/
   Extracted 5000 characters from page
AUDIT VERDICT: ✅ LEGITIMATE — Claims appear substantiated  (Trust Score: 70/100)

 PASS REASON: The description contains verifiable claims, measurable data, or recognized certifications that support its sustainability assertions.

 VAGUE BUZZWORDS DETECTED (3):
   • "natural" — unverifiable without third-party proof
   • "clean" — unverifiable without third-party proof
   • "zero waste" — unverifiable without third-party proof

 POSITIVE EVIDENCE FOUND:
   • Percentage claims: ['60%', '75%', '25%', '24%', '24%', '100%', '100%', '100%', '100%', '100%', '100%', '24%', '99%'] — quantified, verifiable

 CERTIFIED BRAND MATCH:
   • Natura (B-Corp) — recognized certified brand in our database [exact_name]
   • Eileen Fisher (B-Corp) — recognized certified brand in our database [semantic]
   • Thought Clothing (GOTS) — recognized certified brand in

## 📊 Step 9 — Evaluation on Real Test Split

In [62]:
# ── Evaluate ML model on held-out test set ────────────────────────────────────
from sklearn.metrics import confusion_matrix

y_pred_test = ml_pipeline.predict(X_test)

print(f'Accuracy on test set: {accuracy_score(y_test, y_pred_test):.3f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_test, target_names=le.classes_))

print('Confusion Matrix:')
cm = confusion_matrix(y_test, y_pred_test)
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
print(cm_df)

Accuracy on test set: 0.924

Classification Report:
                 precision    recall  f1-score   support

 Evidence-Based       0.82      0.73      0.77        73
Marketing Fluff       0.94      0.97      0.95       349

       accuracy                           0.92       422
      macro avg       0.88      0.85      0.86       422
   weighted avg       0.92      0.92      0.92       422

Confusion Matrix:
                 Evidence-Based  Marketing Fluff
Evidence-Based               53               20
Marketing Fluff              12              337


In [63]:
# ── Show incorrect predictions ─────────────────────────────────────────────────
results_list = []
for txt, true_enc in zip(X_test[:50], y_test[:50]):  # sample 50 for speed
    proba    = ml_pipeline.predict_proba([txt])[0]
    pred_enc = int(np.argmax(proba))
    results_list.append({
        'text'     : txt[:90],
        'true'     : le.inverse_transform([true_enc])[0],
        'predicted': le.inverse_transform([pred_enc])[0],
        'confidence': round(float(np.max(proba)), 2),
        'correct'  : (true_enc == pred_enc)
    })

results_df_eval = pd.DataFrame(results_list)
wrong_df = results_df_eval[~results_df_eval['correct']]
print(f'Wrong predictions (sample of 50): {len(wrong_df)}')
pd.set_option('display.max_colwidth', 90)
print(wrong_df[['text','true','predicted','confidence']].to_string(index=False))

Wrong predictions (sample of 50): 3
                                                                                      text            true       predicted  confidence
                                                     hundreds. www.smartwaternavigator.com Marketing Fluff  Evidence-Based        0.52
                                                                             FATALITIES: 5  Evidence-Based Marketing Fluff        0.68
10 “Managing Water under Uncertainty and Risk”–UNESCO; https://unesdoc.unesco.org/ark:/482  Evidence-Based Marketing Fluff        0.58




```
# This is formatted as code
```

##  Step 10 — Interactive Widget (Colab UI)

In [64]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Widgets ───────────────────────────────────────────────────────────────────
text_box = widgets.Textarea(
    placeholder='Paste product description here...',
    layout=widgets.Layout(width='100%', height='120px')
)
url_box = widgets.Text(
    placeholder='Or enter a product URL: https://...',
    layout=widgets.Layout(width='100%')
)
run_btn = widgets.Button(
    description='🔍 Run Audit',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)
output_area = widgets.Output()

def on_run_clicked(_):
    with output_area:
        clear_output()
        t = text_box.value.strip()
        u = url_box.value.strip()
        if not t and not u:
            print('⚠️  Please enter a product description or URL.')
            return
        try:
            audit(text_input=t, url_input=u, verbose=True)
        except Exception as e:
            print(f'Error: {e}')

run_btn.on_click(on_run_clicked)

# Display
display(
    widgets.VBox([
        widgets.HTML('<h3>🌿 Green-Truth Auditor</h3>'),
        widgets.Label('Product Description:'),
        text_box,
        widgets.Label('— or —'),
        widgets.Label('Product URL:'),
        url_box,
        run_btn,
        output_area
    ])
)

## ✅ Feature Coverage

| Requirement | Status | Implementation |
|---|---|---|
| Text box input | ✅ | `audit(text_input=...)` + ipywidgets UI |
| URL scraper | ✅ | `scrape_url()` using `requests` + `BeautifulSoup` |
| Buzzword detection | ✅ | `detect_buzzwords()` — 28 regex patterns |
| ML classification (Fluff vs Evidence) | ✅ | TF-IDF + LR trained on `Emanuse/greenwashing` |
| Sentence-level classification | ✅ | `sentence_level_classify()` — per-sentence breakdown |
| RAG: Certified brand database | ✅ | 25 B-Corp/GOTS/Fair Trade brands, FAISS semantic search |
| AI Reasoning Summary | ✅ | `generate_reasoning()` — explains exactly why product fails |
| Trust Score (0-100) | ✅ | Calibrated hybrid scoring with 10+ signals |
| Verdict tier | ✅ | 🚩 Greenwashing / ⚠️ Uncertain / ✅ Legitimate |
| Interactive UI | ✅ | ipywidgets with text box + URL box + button |